# 🧑‍💻 AI Databases Demos

![AI Databases event](../assets/ai-databases-event.png)


### ⚙️ Preparations

In [1]:
# Install required packages
# %pip install ipykernel pandas python-dotenv numpy matplotlib seaborn agent-framework azure-mgmt-postgresqlflexibleservers psycopg2-binary

In [2]:
# Import libraries
import os
import json
from dotenv import load_dotenv
import pandas as pd
import textwrap
import re
from IPython.display import Markdown, display

# Load environment variables from .env file
load_dotenv(override=True)
print("✅ Environment variables loaded successfully!")

✅ Environment variables loaded successfully!


In [3]:
# Login using AzureCliCredentials and interactively with browser
from azure.identity import AzureDeveloperCliCredential

credential = AzureDeveloperCliCredential()

print("✅ Azure Developer CLI credentials loaded successfully!", credential.tenant_id)

✅ Azure Developer CLI credentials loaded successfully! 


In [4]:
# Show which identity the credential is actually using (decodes the token claims locally)
import base64

token = credential.get_token("https://management.azure.com/.default").token
payload = token.split(".")[1]
claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))

print(" 🔒 Signed in as:", claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username") or claims.get("appid"))


 🔒 Signed in as: hosseinzahed@MngEnvMCAP302565.onmicrosoft.com


In [5]:
# Load the books dataset
books_df = pd.read_csv('../data/books.csv')

print("✅ Books dataset loaded successfully!", "Number of records:", len(books_df))

✅ Books dataset loaded successfully! Number of records: 6810


In [6]:
# List column names
print("📊 Columns in the dataset:\n", textwrap.fill(", ".join(books_df.columns.tolist()), width=80))

📊 Columns in the dataset:
 isbn13, isbn10, title, subtitle, authors, categories, thumbnail, description,
published_year, average_rating, num_pages, ratings_count


In [7]:
# Show a few records from the dataset for title, authors, categories, and description
print("📚 Sample records from the dataset:")
books_df[['isbn13', 'title', 'authors', 'categories', 'description']].head()

📚 Sample records from the dataset:


,isbn13,title,authors,categories,description
0,9780002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...
1,9780002261982,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...
2,9780006163831,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...
3,9780006178736,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b..."
4,9780006280897,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...


In [8]:
# Show the description of a specific book by index
book_index = 0  # Change this index to view a different book
sample_title = books_df.loc[book_index, 'title']
sample_description = books_df.loc[book_index, 'description']
print(f"📘 Title: {sample_title}")
print(f"📝 Description: \n{textwrap.fill(sample_description, width=100)}")

📘 Title: Gilead
📝 Description: 
A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an
astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and
the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end
of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the
young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift
between his grandfather and his father: the elder, an angry visionary who fought for the
abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake,
Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for
forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and
truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the 

---
---

##  Cosmos DB Demos
<img src="../assets/Azure-Cosmos-DB.png" alt="Cosmos DB Demos" width="150">


In [9]:
# Prepare the Cosmos DB configurations
from azure.cosmos import CosmosClient, PartitionKey

COSMOS_DB_ENDPOINT = os.getenv("COSMOS_DB_ENDPOINT")
COSMOS_DB_DATABASE = os.getenv("COSMOS_DB_DATABASE")
COSMOS_DB_CONTAINER = os.getenv("COSMOS_DB_CONTAINER")

# Create a CosmosClient instance using the endpoint and credentials
cosmos_client = CosmosClient(COSMOS_DB_ENDPOINT, credential=credential)

# Create a database if it doesn't exist
cosmos_database = cosmos_client.create_database_if_not_exists(id=COSMOS_DB_DATABASE)

print(f"✅ Cosmos database '{COSMOS_DB_DATABASE}' created or already exists.")    

✅ Cosmos database 'books-db' created or already exists.


In [10]:
# Create a container in the database if it doesn't exist

# Define vector embedding policy
vector_embedding_policy = {
    "vectorEmbeddings": [
        {
            "path": "/embeddings",   # JSON path to the vector field
            "dataType": "float32",  # Supported: float32
            "dimensions": 1536,     # Example: OpenAI embedding size
            "distanceFunction": "cosine"  # Supported: cosine, euclidean, dotproduct
        }
    ]
}

# Define full text policy, required by FullTextScore and hybrid search
full_text_policy = {
    "defaultLanguage": "en-US",
    "fullTextPaths": [
        {"path": "/title", "language": "en-US"},
        {"path": "/description", "language": "en-US"}
    ]
}

# Define indexing policy
indexing_policy = {
    "indexingMode": "consistent",
    "automatic": True,
    "includedPaths": [
        {"path": "/*"}  # Index all properties
    ],
    "vectorIndexes": [
      {
          "path": "/embeddings",
          "type": "quantizedFlat" # Supported: flat, quantizedFlat, diskANN
      }
    ],
    "fullTextIndexes": [
        {"path": "/title"},
        {"path": "/description"}
    ],
    "excludedPaths": [
        {"path": "/_etag/?"}  # Exclude system property
    ]
}

# Create container with policies
cosmos_container = cosmos_database.create_container_if_not_exists(
    id=COSMOS_DB_CONTAINER,
    partition_key=PartitionKey(path="/id"),  # Example partition key
    indexing_policy=indexing_policy,
    vector_embedding_policy=vector_embedding_policy,
    full_text_policy=full_text_policy
)

print(f"✅ Container '{COSMOS_DB_CONTAINER}' created with vector, full text, and indexing policies.")


✅ Container 'books' created with vector, full text, and indexing policies.


<img src="../assets/cosmosdb-vector-policy.png" alt="Vector Policy" width="300">
<img src="../assets/cosmosdb-full-text-policy.png" alt="Full Text Policy" width="300">

In [11]:
# Create an embedding client for the Foundry service
from agent_framework.foundry import FoundryEmbeddingClient
from azure.ai.inference.aio import EmbeddingsClient

FOUNDRY_MODELS_ENDPOINT = os.getenv("FOUNDRY_MODELS_ENDPOINT")
FOUNDRY_EMBEDDING_MODEL = os.getenv("FOUNDRY_EMBEDDING_MODEL")

# The inference SDK defaults to the ml.azure.com audience, which an AI Services endpoint rejects.
text_client = EmbeddingsClient(
    endpoint=FOUNDRY_MODELS_ENDPOINT,
    credential=credential,
    credential_scopes=["https://cognitiveservices.azure.com/.default"],
)

embedding_client = FoundryEmbeddingClient(
    endpoint=FOUNDRY_MODELS_ENDPOINT,
    model=FOUNDRY_EMBEDDING_MODEL,
    credential=credential,
    text_client=text_client,
)

print(f"🤖 Embedding model: {embedding_client.model}")
# Get the embedding for the sample description
sample_embedding_response = await embedding_client.get_embeddings([sample_description])

# Extract the embedding vector from the response
sample_embedding = sample_embedding_response[0]
print(f"🔢 Sample vector dimensions: {sample_embedding.dimensions}")
print(f"🧮 Sample vector:{textwrap.fill(str(sample_embedding.vector[0:10]), width=70)}")

🤖 Embedding model: text-embedding-3-large
🔢 Sample vector dimensions: 3072
🧮 Sample vector:[-0.005535125732421875, 0.0010805130004882812, -0.01480865478515625,
0.0215911865234375, -0.060882568359375, -0.0086822509765625,
-0.015228271484375, -0.0216217041015625, -0.005779266357421875,
0.0278472900390625]


### ✍️ Data Insertion to Cosmos DB

In [12]:
# Insert books into the Cosmos DB container with embeddings
from collections import Counter

# Check whether at least one record exists before inserting
existing_items = next(iter(cosmos_container.read_all_items(max_item_count=1)), None)

# If container is not empty, skip insertion to avoid duplicates
if existing_items:
    print(
        f"⚠️ Container '{COSMOS_DB_CONTAINER}' already has data. Skipping insertion to avoid duplicates.")
else:
    # Limit to first 1000 books for demonstration
    books_batch = books_df[:1000]

    inserted_count = 0
    failure_reasons = Counter()

    for index, row in books_batch.iterrows():
        try:
            # Get the embedding for the book description
            book_title = row['title']
            book_description = row['description']
            embedding_context = f"{book_title}: {book_description}"

            book_embedding_response = await embedding_client.get_embeddings([embedding_context])
            book_embedding = book_embedding_response[0]

            # Create a document to insert into Cosmos DB
            document = {
                "id": str(row['isbn13']),  # Ensure the ID is a string
                "title": book_title,
                "authors": row['authors'],
                "categories": row['categories'],
                "description": book_description,
                "embeddings": book_embedding.vector
            }

            # Validate the document to ensure it can be serialized to JSON
            json.dumps(document)

            # Insert the document into the Cosmos DB container
            cosmos_container.upsert_item(document)
            inserted_count += 1

        except Exception as e:
            failure_reasons[type(e).__name__] += 1
            continue  # Skip to the next book in case of an error

    skipped_count = sum(failure_reasons.values())
    print(f"📊 Processed {len(books_batch)} books.")
    print(f"✅ Inserted {inserted_count} books into the Cosmos DB container with embeddings.")
    if skipped_count:
        reasons = ", ".join(f"{name} x{count}" for name, count in failure_reasons.most_common())
        print(f"⚠️ Skipped {skipped_count} books ({reasons}).")


⚠️ Container 'books' already has data. Skipping insertion to avoid duplicates.


#### 📚 List of documents in books-db
<img src="../assets/cosmosdb-documents.png" alt="Cosmos DB Documents" width="100%">

#### 💲Tokens used to generate embeddings for the description field across 1,000 records.
<img src="../assets/embedding-model.png" alt="Embedding Model" width="100%">

#### ✨ Cosmos DB Features
<img src="../assets/cosmosdb-features.png" alt="Cosmos DB Features" width="100%">

### 🔎 Keyword Search

In [13]:
# Keyword search sample query
search_keyword = "science fiction"

# CONTAINS with the case-insensitive flag matches the keyword anywhere in the title or description
keyword_query = """
SELECT TOP 5 c.id, c.title, c.authors, c.categories, c.description
FROM c
WHERE 
    CONTAINS(c.title, @keyword, true) 
    OR 
    CONTAINS(c.description, @keyword, true)
"""

In [14]:
# Run the keyword search query against the Cosmos DB container
keyword_results = list(cosmos_container.query_items(
    query=keyword_query,
    parameters=[{"name": "@keyword", "value": search_keyword}],
    enable_cross_partition_query=True
))

HIGHLIGHT = "\033[4;30;103m"  # underline + black text on a bright yellow background
RESET = "\033[0m"


def highlight(text, keyword):
    """Underline and highlight keyword matches using ANSI escape codes."""
    # Allow any whitespace between words so matches split across wrapped lines still highlight
    pattern = r"\s+".join(re.escape(word) for word in keyword.split())
    return re.sub(pattern, lambda m: f"{HIGHLIGHT}{m.group(0)}{RESET}", text, flags=re.IGNORECASE)


print(f"🔍 Keyword search for '{search_keyword}' returned {len(keyword_results)} results:\n")
for item in keyword_results:
    # Wrap before highlighting so the escape codes don't count towards the line width
    wrapped_description = textwrap.fill(
        item['description'], width=75, initial_indent='   ', subsequent_indent='   ')
    print(f"📘 {highlight(item['title'], search_keyword)} — {item['authors']}")
    print(f"{highlight(wrapped_description, search_keyword)}\n")


🔍 Keyword search for 'science fiction' returned 4 results:

📘 Gold — Isaac Asimov
   Gold is the final and crowning achievement of the fifty-year career of
   science fiction's transcendent genius, the world-famous author who
   defined the field of science fiction for its practitioners, its millions
   of readers, and the world at large. The first section contains stories
   that range from the humorous to the profound, at the heart of which is
   the title story, "Gold," a moving and revealing drama about a writer who
   gambles everything on a chance at immortality: a gamble Asimov himself
   made -- and won. The second section contains the grand master's
   ruminations on the SF genre itself. And the final section is comprised
   of Asimov's thoughts on the craft and writing of science fiction.

📘 Racso and the Rats of NIMH — Jane Leslie Conly
   ‘Racso, a brash and boastful little rodent, is making his way to Thorn
   Valley, determined to learn how to read and write and become a 

### 🔢 Vector Search

In [15]:
# Vector (semantic) search sample query
search_phrase = "a lighthearted story about friendship and adventure"

# VectorDistance scores each document against the query vector; ORDER BY makes it a nearest-neighbour search
vector_query = """
SELECT TOP 5 c.title, c.authors, c.categories, c.description,
       VectorDistance(c.embeddings, @searchVector) AS similarity_score
FROM c
ORDER BY VectorDistance(c.embeddings, @searchVector)
"""

# Embed the search phrase with the same model used for the stored documents
search_embedding_response = await embedding_client.get_embeddings([search_phrase])
search_vector = search_embedding_response[0].vector

In [16]:
# Run the vector search query against the Cosmos DB container
vector_results = list(cosmos_container.query_items(
    query=vector_query,
    parameters=[{"name": "@searchVector", "value": search_vector}],
    enable_cross_partition_query=True
))

print(f"🧭 Vector search for '{search_phrase}' returned {len(vector_results)} results:\n")
for item in vector_results:
    score = f"{HIGHLIGHT}score: {item['similarity_score']:.4f}{RESET}"
    print(f"📘 {item['title']} — {item['authors']}  ({score})")
    print(textwrap.fill(item['description'], width=75,
          initial_indent='   ', subsequent_indent='   '), "\n")


🧭 Vector search for 'a lighthearted story about friendship and adventure' returned 5 results:

📘 Little House Friends — Heather Henson;Laura Ingalls Wilder  (score: 0.4327)
   Laura Ingalls shares adventures and good times with her friends while
   growing up on the western frontier. 

📘 The Giraffe and the Pelly and Me — Roald Dahl;Quentin Blake  (score: 0.4121)
   A Dahl story in which the giraffe, the pelican and the agile monkey set
   out to prove that they are the best window-cleaning company around. 

📘 The Illustrated Alchemist — Paulo Coelho;Alan R. Clarke;Moebius  (score: 0.4052)
   This fable aims teaches the reader to open their mind, listen to their
   heart and most importantly, follow their dreams. 

📘 Oliver and Albert, Friends Forever — Jean Van Leeuwen  (score: 0.3975)
   Oliver makes friends with Albert, the new boy in class, and they have
   fun together, playing kickball and collecting bugs. By the creators of
   Amanda Pig, Schoolgirl. Reprint. 

📘 Charlotte's Web

### 🔀 Hybrid Search

In [17]:
# Hybrid search: fuse keyword ranking and vector similarity with Reciprocal Rank Fusion (RRF)
hybrid_keywords = ["friendship", "adventure"]

# FullTextScore only accepts literal terms, so they go into the query text (json.dumps escapes them safely)
hybrid_terms = ", ".join(json.dumps(keyword) for keyword in hybrid_keywords)

# RRF (Reciprocal Rank Fusion) combines the ranks of 
# multiple scoring functions to produce a final ranking
hybrid_query = f"""
SELECT TOP 5 c.title, c.authors, c.categories, c.description
FROM c
ORDER BY RANK RRF(
    VectorDistance(c.embeddings, @searchVector),
    FullTextScore(c.description, {hybrid_terms})
)
"""

# Hybrid search with weighted ranking: 
# Give vector similarity twice the weight of keyword ranking
# [2,1] means the vector is weighted 2x, the keyword is weighted 1x
hybrid_weighted_query = f"""
SELECT TOP 5 c.title, c.authors, c.categories, c.description
FROM c
ORDER BY RANK RRF(
    VectorDistance(c.embeddings, @searchVector),    
    FullTextScore(c.description, {hybrid_terms}),
    [2,1]
)
"""

In [18]:
# Reuses the query vector from the vector search cell above
hybrid_results = list(cosmos_container.query_items(
    query=hybrid_query,
    parameters=[{"name": "@searchVector", "value": search_vector}],
    enable_cross_partition_query=True
))

print(f"🔀 Hybrid search for '{search_phrase}' + {hybrid_keywords} returned {len(hybrid_results)} results:\n")
for item in hybrid_results:
    print(f"📘 {item['title']} — {item['authors']}")
    print(textwrap.fill(item['description'], width=75,
          initial_indent='   ', subsequent_indent='   '), "\n")


🔀 Hybrid search for 'a lighthearted story about friendship and adventure' + ['friendship', 'adventure'] returned 5 results:

📘 Little House Friends — Heather Henson;Laura Ingalls Wilder
   Laura Ingalls shares adventures and good times with her friends while
   growing up on the western frontier. 

📘 Pippi Goes on Board — Astrid Lindgren;Louis S. Glanzman
   The further adventures of Pippi and her friends Tommy and Annika. 

📘 Of Mice and Men — John Steinbeck
   The tragic story of the friendship between two migrant workers, George
   and mentally retarded Lenny, and their dream of owning a farm 

📘 Captain Cat — Syd Hoff
   A patriotic feline, Captain Cat springs out of bed whenever the bugle
   sounds and he has more stripes than any of the soldiers. But most of
   all, this young recruit and his best friend Pete know what it really
   takes to make the army a home—friendship. ‘Hoff continues his string of
   hits.’ —BL. ‘Hoff has maintained his deft touch with a title that’s sure
  

In [19]:
# Create an agent that can answer questions about the books dataset 
# using the Foundry chat model
from agent_framework.foundry import FoundryChatClient
from agent_framework import Agent

FOUNDRY_CHAT_MODEL = os.getenv("FOUNDRY_CHAT_MODEL")
FOUNDRY_SERVICES_ENDPOINT = os.getenv("FOUNDRY_SERVICES_ENDPOINT")

chat_client = FoundryChatClient(
    project_endpoint=FOUNDRY_SERVICES_ENDPOINT,
    model=FOUNDRY_CHAT_MODEL,
    credential=credential)

agent = Agent(
        client=chat_client,
        name="LibraryAssistant",
        instructions="You're a friendly library assistant. You can answer questions about the books dataset, provide recommendations, and summarize book descriptions. Keep your answers brief and informative.",
    )

prompt = "Recommend a few books about friendship and adventure"

enriched_prompt = f"{prompt}\n\n{hybrid_results}"

result = await agent.run(enriched_prompt)

# print() writes plain text, so the model's markdown has to be displayed to render
display(Markdown(f"**🤖 Agent:**\n\n{result}"))


**🤖 Agent:**

Here are a few good picks about friendship and adventure:

- **Little House Friends** — Laura Ingalls shares adventures and good times with her friends on the western frontier.
- **Pippi Goes on Board** — More adventures with Pippi and her friends Tommy and Annika.
- **Captain Cat** — A fun story where Captain Cat and his best friend Pete learn what friendship really means.
- **Five Children and It** — Five children have a series of magical adventures after finding a sand-fairy.

If you want, I can also suggest the most **classic**, **funniest**, or **best for younger readers** among these.

---
---

##  PostgreSQL Demos
<img src="../assets/Azure-Database-PostgreSQL-Server.png" alt="PostgreSQL" width="150">


In [20]:
# Prepare the PostgreSQL configurations
from azure.mgmt.postgresqlflexibleservers import PostgreSQLManagementClient
from azure.mgmt.postgresqlflexibleservers.models import Database

RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID")
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_TABLE = os.getenv("POSTGRES_TABLE")

# The server name is the first label of the host (e.g. my-server.postgres.database.azure.com)
POSTGRES_SERVER = POSTGRES_HOST.split('.')[0]

# This is an ARM control-plane client, so it must talk to management.azure.com (the SDK default),
# not to the server host on port 5432, which speaks the Postgres wire protocol instead of HTTPS.
postgres_client = PostgreSQLManagementClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID
)

# Create the PostgreSQL database if it doesn't exist
response = postgres_client.databases.begin_create(
    resource_group_name=RESOURCE_GROUP,
    server_name=POSTGRES_SERVER,
    database_name=POSTGRES_DB,
    parameters=Database(charset="utf8", collation="en_US.utf8"),
).result()  # Wait for the operation to complete

print(f"✅ PostgreSQL database '{POSTGRES_DB}' created or already exists.")

✅ PostgreSQL database 'books-db' created or already exists.


In [21]:
# Check PostgreSQL extensions to ensure the required extensions are available
extensions_config = postgres_client.configurations.get(
    resource_group_name=RESOURCE_GROUP,
    server_name=POSTGRES_SERVER,
    configuration_name="azure.extensions",
)

# List the allow-listed extensions from the configuration
allow_listed = [name for name in (
    extensions_config.value or "").split(",") if name]

print(f"🧩 Enabled Azure extensions on the PostgreSQL server ({len(allow_listed)}):")
for name in sorted(allow_listed):
    print(f"   • {name}")

🧩 Enabled Azure extensions on the PostgreSQL server (4):
   • age
   • azure_ai
   • pg_diskann
   • vector


#### PostgreSQL Azure Extensions
<img src="../assets/postgres-az-extensions.png" alt="PostgreSQL Azure Extensions" width="100%">

In [22]:
# Create a Postgres table for the books dataset if it doesn't exist
import psycopg2
from psycopg2 import sql

# Entra ID auth: the access token is the password, and the login name is the signed-in principal
pg_token = credential.get_token(
    "https://ossrdbms-aad.database.windows.net/.default").token

# Connect to the PostgreSQL database using psycopg2 with Entra ID authentication
postgres_connection = psycopg2.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    dbname=POSTGRES_DB,
    user=claims["upn"],
    password=pg_token,
    sslmode="require",
    # libpq 18 defaults to gssencmode='prefer'; the Azure gateway drops the GSSENCRequest packet
    gssencmode="disable",
)
postgres_connection.autocommit = True

# Create the table if it doesn't exist
with postgres_connection.cursor() as cursor:
    # Identifier() quotes the env-provided table name so it can't inject SQL
    cursor.execute(sql.SQL("""
        CREATE TABLE IF NOT EXISTS {table} (
            id          TEXT PRIMARY KEY,
            title       TEXT,
            authors     TEXT,
            categories  TEXT,
            description TEXT
        );
    """).format(table=sql.Identifier(POSTGRES_TABLE)))

print(f"✅ Table '{POSTGRES_TABLE}' created or already exists in '{POSTGRES_DB}'.")


✅ Table 'books' created or already exists in 'books-db'.


In [23]:
# Insert books into the PostgreSQL table
inserted_count = 0
books_batch = books_df[:1000]  # Limit to first 1000 books for demonstration

# Check whether at least one record exists before inserting
with postgres_connection.cursor() as cursor:
    cursor.execute(sql.SQL("SELECT 1 FROM {table} LIMIT 1;").format(
        table=sql.Identifier(POSTGRES_TABLE)))
    existing_record = cursor.fetchone()
    
    if existing_record:
        print(
            f"⚠️ Table '{POSTGRES_TABLE}' already has data. Skipping insertion to avoid duplicates.")
    else:
        for index, row in books_batch.iterrows():
            try:
                with postgres_connection.cursor() as cursor:
                    cursor.execute(sql.SQL("""
                        INSERT INTO {table} (id, title, authors, categories, description)
                        VALUES (%s, %s, %s, %s, %s)
                        ON CONFLICT (id) DO NOTHING;
                    """).format(table=sql.Identifier(POSTGRES_TABLE)),
                        (str(row['isbn13']), 
                         row['title'], 
                         row['authors'], 
                         row['categories'], 
                         row['description']))
                    inserted_count += 1
            except Exception as e:
                print(f"⚠️ Failed to insert book '{row['title']}': {e}")
                continue  # Skip to the next book in case of an error

        print(f"✅ Inserted {inserted_count} books into the PostgreSQL table.")


⚠️ Table 'books' already has data. Skipping insertion to avoid duplicates.


#### 📚 List of records in books-db
<img src="../assets/postgres-records.png" alt="PostgreSQL List of Records" width="100%">

In [24]:
# Configure the PostgreSQL server to Foundry for generating embeddings
# The azure_ai extension calls the OpenAI-form endpoint, not the .services.ai.azure.com one
FOUNDRY_OPENAI_ENDPOINT = os.getenv("FOUNDRY_OPENAI_ENDPOINT")
FOUNDRY_EMBEDDING_SMALL_MODEL = os.getenv("FOUNDRY_EMBEDDING_SMALL_MODEL")

with postgres_connection.cursor() as cursor:
    cursor.execute("CREATE EXTENSION IF NOT EXISTS azure_ai;")

    # Managed identity keeps an API key out of the extension's settings table
    cursor.execute(
        "SELECT azure_ai.set_setting('azure_openai.auth_type', 'managed-identity');")
    cursor.execute(
        "SELECT azure_ai.set_setting('azure_openai.endpoint', %s);", (FOUNDRY_OPENAI_ENDPOINT,))

    # Verify the settings were applied correctly
    cursor.execute("""
        SELECT azure_ai.get_setting('azure_openai.auth_type'),
               azure_ai.get_setting('azure_openai.endpoint');
    """)
    auth_type, endpoint = cursor.fetchone()

print("✅ PostgreSQL server configured for Foundry embeddings.")
print(f"🔐 Auth type: {auth_type}")
print(f"🌐 Endpoint:  {endpoint}")
print(f"🤖 Embeddings model: {FOUNDRY_EMBEDDING_SMALL_MODEL}")


✅ PostgreSQL server configured for Foundry embeddings.
🔐 Auth type: managed-identity
🌐 Endpoint:  https://aif-msf4glwsl2lxo.openai.azure.com
🤖 Embeddings model: text-embedding-3-small


#### ✨ PostgreSQL + Azure OpenAI Integration
<img src="../assets/postgres-aoai.png" alt="Postgres and Azure OpenAI" width="100%">

In [25]:
# Add a vector column to the PostgreSQL table for storing embeddings
import time

# DiskANN caps out at 2000 dimensions, so this demo uses the 1536-dimension small model
FOUNDRY_EMBEDDING_SMALL_MODEL = os.getenv("FOUNDRY_EMBEDDING_SMALL_MODEL")

EMBEDDING_DIMENSIONS = 1536    # text-embedding-3-small
EMBEDDING_CHUNK_SIZE = 50      # rows per statement, so a throttled call costs one chunk instead of the whole run
EMBEDDING_PAUSE_SECONDS = 60   # let the per-minute quota refill between chunks

with postgres_connection.cursor() as cursor:

    # pgvector provides the `vector` type; pg_diskann provides the ANN index
    cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
    cursor.execute("CREATE EXTENSION IF NOT EXISTS pg_diskann CASCADE;")

    cursor.execute(sql.SQL("""
        ALTER TABLE {table}
        ADD COLUMN IF NOT EXISTS embeddings vector({dims});
    """).format(table=sql.Identifier(POSTGRES_TABLE),
                dims=sql.Literal(EMBEDDING_DIMENSIONS)))

    # Resize the column if it was created for a different embedding model (requires the column to be NULL)
    cursor.execute(sql.SQL("""
        ALTER TABLE {table}
        ALTER COLUMN embeddings TYPE vector({dims});
    """).format(table=sql.Identifier(POSTGRES_TABLE),
                dims=sql.Literal(EMBEDDING_DIMENSIONS)))

    cursor.execute(sql.SQL("""
        SELECT COUNT(*) FROM {table}
        WHERE embeddings IS NULL AND description IS NOT NULL;
    """).format(table=sql.Identifier(POSTGRES_TABLE)))
    pending_count = cursor.fetchone()[0]
    print(f"🧮 {pending_count} records need embeddings.")

    # Targeting only NULL embeddings makes the cell resumable: re-run it after a rate-limit failure
    # and it picks up where it stopped. autocommit means each chunk is durable on its own.
    embedded_count = 0
    backoff_seconds = EMBEDDING_PAUSE_SECONDS

    while True:
        try:
            cursor.execute(sql.SQL("""
                UPDATE {table}
                SET embeddings = azure_openai.create_embeddings(
                        %s, description,
                        max_attempts => 5,
                        retry_delay_ms => 10000
                    )::vector
                WHERE id IN (
                    SELECT id FROM {table}
                    WHERE embeddings IS NULL AND description IS NOT NULL
                    LIMIT %s
                );
            """).format(table=sql.Identifier(POSTGRES_TABLE)),
                (FOUNDRY_EMBEDDING_SMALL_MODEL, EMBEDDING_CHUNK_SIZE))
        except psycopg2.Error as error:
            if "RateLimitReached" not in str(error):
                raise
            # The quota is per minute, so back off further each time instead of retrying at a fixed delay
            print(f"   ⏳ throttled — waiting {backoff_seconds}s before retrying this chunk")
            time.sleep(backoff_seconds)
            backoff_seconds = min(backoff_seconds * 2, 120)
            continue

        if cursor.rowcount == 0:
            break

        embedded_count += cursor.rowcount
        backoff_seconds = EMBEDDING_PAUSE_SECONDS
        print(f"   … {embedded_count}/{pending_count} embedded")
        time.sleep(EMBEDDING_PAUSE_SECONDS)

    print(f"✅ Embeddings generated for '{POSTGRES_TABLE}' using '{FOUNDRY_EMBEDDING_SMALL_MODEL}'.")

    # Build the index after the column is populated so DiskANN sees the real data distribution
    cursor.execute(sql.SQL("""
        CREATE INDEX IF NOT EXISTS {index} ON {table}
        USING diskann (embeddings vector_cosine_ops);
    """).format(index=sql.Identifier(f"{POSTGRES_TABLE}_embeddings_diskann"),
                table=sql.Identifier(POSTGRES_TABLE)))

    print(f"✅ DiskANN index ready on '{POSTGRES_TABLE}.embeddings' ({EMBEDDING_DIMENSIONS} dimensions).")


🧮 850 records need embeddings.
   … 50/850 embedded
   … 100/850 embedded
   … 150/850 embedded
   … 200/850 embedded
   … 250/850 embedded
   … 300/850 embedded
   … 350/850 embedded
   … 400/850 embedded
   … 450/850 embedded
   … 500/850 embedded
   … 550/850 embedded
   … 600/850 embedded
   … 650/850 embedded
   … 700/850 embedded
   … 750/850 embedded
   … 800/850 embedded
   … 850/850 embedded
✅ Embeddings generated for 'books' using 'text-embedding-3-small'.
✅ DiskANN index ready on 'books.embeddings' (1536 dimensions).
